[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/14_bits.ipynb)

*DSA Interview Handbook · Notebook 14 of 14*

# Bit Manipulation

Work on the binary digits directly: XOR cancelling, n & (n − 1), masks and per-bit counting.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A number is a row of switches, and bit tricks flip many of them in one operation.

**What interviewers really test here:** XOR tricks, masks, and n & (n − 1).

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Binary numbers *(Basics)*
- [ ] 2. The six operators *(Basics)*
- [ ] 3. Working with one bit *(Core)*
- [ ] 4. XOR, the cancelling operator *(Core)*
- [ ] 5. Counting and comparing bits *(Core)*
- [ ] 6. Negative numbers and arithmetic with bits *(Advanced)*
- [ ] 7. Per-bit thinking, prefix XOR and subsets *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

**Bit manipulation** is the act of using bitwise operators to read, set, clear or combine the individual binary digits (**bits**) of integers. Because processors execute bitwise operations in a single instruction, these techniques are extremely fast and memory-efficient.



A **bit** is 0 or 1. A **byte** is 8 bits. A **word** is the processor's natural integer size (32 or 64 bits). The **least significant bit (LSB)** is bit 0 (worth 1); the **most significant bit (MSB)** is the highest position.

### Number systems

| System | Base | Digits | 13 written as | Python |
| --- | --- | --- | --- | --- |
| Binary | 2 | 0–1 | 1101 | `0b1101`, `bin(13)` |
| Octal | 8 | 0–7 | 15 | `0o15`, `oct(13)` |
| Decimal | 10 | 0–9 | 13 | `13` |
| Hexadecimal | 16 | 0–9, A–F | D | `0xD`, `hex(13)` |



**Binary → decimal**: sum bitᵢ × 2ⁱ. **Decimal → binary**: repeatedly divide by 2 and read the remainders bottom-up. Each hex digit is exactly 4 bits.

### Bitwise operators and truth table

| A | B | A & B | A \| B | A ^ B |
| --- | --- | --- | --- | --- |
| 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 | 1 |
| 1 | 0 | 0 | 1 | 1 |
| 1 | 1 | 1 | 1 | 0 |



| Operator | Name | Effect |
| --- | --- | --- |
| `~x` | NOT | flips every bit; in two's complement `~x = −x − 1` |
| `x << k` | left shift | x × 2ᵏ |
| `x >> k` | right shift | ⌊x / 2ᵏ⌋ (arithmetic shift keeps the sign for negatives) |



#### Algebraic laws of XOR




- Identity: `x ^ 0 = x`; self-inverse: `x ^ x = 0`
- Commutative and associative: order and grouping don't matter
- Therefore `a ^ b ^ a = b`, and `a ^ b = c ⇔ a = b ^ c`

### Representing negative numbers

| Scheme | −5 in 8 bits | Notes |
| --- | --- | --- |
| Sign-magnitude | 1000 0101 | MSB is the sign; two zeros (+0, −0) |
| One's complement | 1111 1010 | flip all bits; still two zeros |
| **Two's complement** | 1111 1011 | flip all bits and add 1; one zero; used by all modern CPUs |



In n-bit two's complement the range is −2ⁿ⁻¹ … 2ⁿ⁻¹ − 1 (for 32 bits: −2,147,483,648 … 2,147,483,647). Addition and subtraction work the same for signed and unsigned values, which is why hardware uses it.



**Overflow** happens when a result is outside the range and wraps around (e.g. 2³¹ − 1 + 1 becomes −2³¹ in 32 bits). Python integers are arbitrary precision and never overflow — mask with `& 0xFFFFFFFF` to simulate 32-bit behaviour.

### Standard bit tricks

| Task | Expression |
| --- | --- |
| Check if bit i is set | `(n >> i) & 1` or `n & (1 << i) != 0` |
| Set / clear / toggle bit i | `n \| (1 << i)` / `n & ~(1 << i)` / `n ^ (1 << i)` |
| Is n even? | `n & 1 == 0` |
| Is n a power of 2? | `n > 0 and n & (n − 1) == 0` |
| Clear lowest set bit | `n & (n − 1)` |
| Isolate lowest set bit | `n & −n` |
| Count set bits (Kernighan) | `while n: n &= n − 1; count += 1` |
| Mask of the lowest k bits | `(1 << k) − 1` |
| Swap without temp | `a ^= b; b ^= a; a ^= b` |
| Iterate all subsets of n items | `for mask in range(1 << n)` |
| Iterate submasks of mask | `s = mask; while s: …; s = (s − 1) & mask` |

### Advantages, disadvantages and applications

**👍 Pros**

##### Advantages

- Single-instruction speed
- Compact storage: 64 booleans in one integer (bitsets)
- Elegant O(1)-space solutions (XOR tricks)



**👎 Cons**

##### Disadvantages

- Hard to read; easy to make precedence mistakes
- Sign and overflow behaviour differs between languages
- Gains are often irrelevant next to algorithmic improvements





#### Applications




- Permissions and feature flags (Unix file modes like `0o755`).
- Bitsets / Bloom filters for compact membership tests.
- Graphics: colours packed as 0xRRGGBB; masks and blending.
- Networking: IP subnet masks, checksums, protocol headers.
- Cryptography and hashing (XOR, rotations, shifts).
- Compression and error-detecting codes (parity, Hamming codes, Gray codes in encoders).
- Bitmask DP and subset enumeration in algorithms.

---
# Part 2: Lessons, step by step

## Lesson 1: Binary numbers

*Level: Basics*

Computers store numbers in **base 2**: each digit (a **bit**) is 0 or 1, and each position is worth double the one to its right.



```text
bit position:   3   2   1   0
worth:          8   4   2   1
13 in binary:   1   1   0   1     → 8 + 4 + 1 = 13
```



To convert a number to binary, repeatedly divide by 2; the remainders, read from last to first, are the bits. In Python: `bin(13)` → `'0b1101'`, `int('1101', 2)` → 13, `format(13, '08b')` → `'00001101'`.



An n-bit number ranges from 0 to 2ⁿ − 1, so 32 bits hold about 4 billion values.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def to_binary_lesson():
    n = 13
    bits, x = [], n
    while x:
        bits.append(x % 2)
        x //= 2
    return bits


to_binary_lesson()

[1, 0, 1, 1]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. 13 ÷ 2 = 6, remainder 1 → next bit (right to left)
2. 6 ÷ 2 = 3, remainder 0 → next bit (right to left)
3. 3 ÷ 2 = 1, remainder 1 → next bit (right to left)
4. 1 ÷ 2 = 0, remainder 1 → next bit (right to left)
5. Read the remainders backwards: 13 = 1101

</details>


**✏️ Your turn:** you now have what you need for [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 2: The six operators

*Level: Basics*

| Op | Name | Rule per bit | 5 (0101) with 3 (0011) |
| --- | --- | --- | --- |
| `&` | AND | 1 if both are 1 | 0001 = 1 |
| `\|` | OR | 1 if either is 1 | 0111 = 7 |
| `^` | XOR | 1 if they differ | 0110 = 6 |
| `~` | NOT | flip every bit | `~5 = −6` in Python |
| `<<` | left shift | move bits left, add 0s | `5 << 1` = 1010 = 10 |
| `>>` | right shift | move bits right, drop the last | `5 >> 1` = 0010 = 2 |



`x << k` multiplies by 2ᵏ and `x >> k` divides by 2ᵏ (rounding down). The topic's quick-reference animation steps through all of these.

## Lesson 3: Working with one bit

*Level: Core*

Build a **mask** with a single 1 at position i: `1 << i`. Then:



| Goal | Code |
| --- | --- |
| read bit i | `(n >> i) & 1` |
| set bit i to 1 | `n \| (1 << i)` |
| clear bit i to 0 | `n & ~(1 << i)` |
| toggle bit i | `n ^ (1 << i)` |
| lowest set bit only | `n & -n` |
| remove lowest set bit | `n & (n - 1)` |



`n & (n − 1)` works because subtracting 1 flips the lowest 1 to 0 and every 0 below it to 1; the AND clears them all. It counts set bits in as many steps as there are 1s, and `n & (n − 1) == 0` means n is a power of two.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def single_bit_lesson():
    n, i = 0b1010, 2
    s = n | (1 << i)
    c = s & ~(1 << i)
    t = n ^ (1 << 1)
    return t


single_bit_lesson()

8

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Read: (n &gt;&gt; 2) & 1 = 0
2. Set: n | mask turns bit 2 on
3. Clear: n & ~mask turns it off again
4. Toggle bit 1: n ^ (1 &lt;&lt; 1) flips just that bit

</details>


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q5](#q5), [Q6](#q6), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 4: XOR, the cancelling operator

*Level: Core*

XOR has three properties that solve a whole class of problems:




- `x ^ x = 0` — equal values cancel.
- `x ^ 0 = x` — zero changes nothing.
- Order doesn't matter: `a ^ b ^ a = b`.



So XOR-ing a list where everything appears twice except one value leaves exactly that value — O(n) time, O(1) memory. The same works for characters (find the extra letter) and for missing numbers (XOR indices with values).



**Two** unique values? XOR all to get `a ^ b`; any 1-bit in it is a place where a and b differ. Split the numbers by that bit and XOR each group.



**Three** copies of every value except one? Count each bit position modulo 3 (or use the compact `ones`/`twos` masks).

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q9](#q9), [Q12](#q12), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 5: Counting and comparing bits

*Level: Core*

- **Hamming distance** between x and y = number of 1s in `x ^ y` (the differing positions).
- **Counting bits for 0..n**: `ans[i] = ans[i >> 1] + (i & 1)` — i has i's bits shifted plus its last bit.
- **Bit flips to convert** start → goal = popcount(start ^ goal).
- **Minimum flips for a | b = c**: handle each bit position as a small case analysis.



Python helpers: `bin(x).count("1")`, or `x.bit_count()` in Python 3.10+. In interviews, also show the `n & (n − 1)` loop.

**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q10](#q10), [Q17](#q17), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 6: Negative numbers and arithmetic with bits

*Level: Advanced*

#### Two's complement



Fixed-width integers store −x as `~x + 1` (flip all bits, add one). In 8 bits, −5 is `11111011`. The top bit acts as the sign.



Python integers have no fixed width — conceptually a negative has infinitely many leading 1s. For "32-bit" problems, keep values inside a mask `0xFFFFFFFF` and convert back at the end: `x if x < 2**31 else x - 2**32`.



#### Adding without +



`a ^ b` adds without carrying; `(a & b) << 1` is the carry. Repeat until there is no carry.



#### Dividing without /



Subtract the biggest `divisor << k` that fits, add `1 << k` to the quotient, repeat — O(log² n) instead of O(quotient).



#### Reversing bits



Swap halves with masks: 16-bit halves, then bytes, then nibbles, pairs and single bits — 5 steps for 32 bits.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def twos_complement_lesson():
    x, W = 5, 8
    flipped = ~x & 0xFF
    neg = (flipped + 1) & 0xFF
    back = neg - 256 if neg >= 128 else neg
    return back


twos_complement_lesson()

-5

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. 5 in 8 bits
2. Flip every bit (~x)
3. Add 1 → this bit pattern means −5
4. Read back as signed: top bit set, so 251 − 256 = -5

</details>


**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q16](#q16), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 7: Per-bit thinking, prefix XOR and subsets

*Level: Advanced*

#### Count each bit separately



For "sum over all pairs" questions, bits don't affect each other. Total Hamming distance: if k numbers have bit b set, that bit differs in `k × (n − k)` pairs. Sum over the 32 bits → O(32n) instead of O(n²).



#### Prefix XOR



Like prefix sums: `pre[i]` = XOR of the first i values, and `xor(l..r) = pre[r + 1] ^ pre[l]` because the shared part cancels. A range XOR is 0 exactly when two prefixes are equal.



#### Masks as subsets



An n-bit mask picks a subset of n items: bit i set = item i included. `for mask in range(1 << n)` visits all 2ⁿ subsets — the basis of bitmask DP.



#### Common prefix



AND of a whole range keeps only the common binary prefix of its ends. Gray code `i ^ (i >> 1)` changes one bit per step.

**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q8](#q8), [Q19](#q19), [Q18](#q18), [Q14](#q14), [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Single Number

🟢 Warm-up · *XOR cancels pairs*

Every element appears twice except one. Find it in O(n) time and O(1) space.

**Examples**

- `nums = [2, 2, 1]` → `1`
- `nums = [4, 1, 2, 1, 2]` → `4`
- `nums = [1]` → `1`

In [ ]:
def single_number(nums):
    """
    Every element appears twice except one. Find it in O(n) time and O(1) space.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(single_number, [
    (([2, 2, 1],), 1),
    (([4, 1, 2, 1, 2],), 4),
    (([1],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count occurrences in a dict and return the value seen once.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

XOR all numbers: x ^ x = 0 and 0 ^ y = y, and order doesn't matter, so every pair cancels and only the single number is left.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count occurrences in a dict and return the value seen once.

```python
def single_number(nums):
    count = {}
    for x in nums:
        count[x] = count.get(x, 0) + 1
    return next(x for x, c in count.items() if c == 1)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** XOR all numbers: x ^ x = 0 and 0 ^ y = y, and order doesn't matter, so every pair cancels and only the single number is left.

```python
def single_number(nums):
    acc = 0
    for i, x in enumerate(nums):
        acc ^= x
    return acc
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count occurrences in a dict and return the value seen once. That costs O(n) time and O(n) space."

"I can do better: XOR all numbers: x ^ x = 0 and 0 ^ y = y, and order doesn't matter, so every pair cancels and only the single number is left. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [2, 2, 1]`

1. acc ^= 2 → 2
2. acc ^= 2 → 0
3. acc ^= 1 → 1

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/single-number)

<a id="q2"></a>
### Q2. Number of 1 Bits

🟢 Warm-up · *n & (n − 1)*

Return the number of set bits in the binary form of a positive integer.

**Examples**

- `n = 11` → `3`
- `n = 128` → `1`
- `n = 2147483645` → `30`

In [ ]:
def hamming_weight(n):
    """
    Return the number of set bits in the binary form of a positive integer.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hamming_weight, [
    ((11,), 3),
    ((128,), 1),
    ((2147483645,), 30),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check all 32 bit positions one by one.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

n & (n − 1) removes the lowest set bit, so loop until n is 0 — only as many steps as there are 1s.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Check all 32 bit positions one by one.

```python
def hamming_weight(n):
    return sum((n >> i) & 1 for i in range(32))
```

**Cost:** O(32) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** n & (n − 1) removes the lowest set bit, so loop until n is 0 — only as many steps as there are 1s.

```python
def hamming_weight(n):
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count
```

**Cost:** O(set bits) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check all 32 bit positions one by one. That costs O(32) time and O(1) space."

"I can do better: n & (n − 1) removes the lowest set bit, so loop until n is 0 — only as many steps as there are 1s. That's O(set bits) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 44`

1. n = 44: clear its lowest set bit
2. n = 40: clear its lowest set bit
3. n = 32: clear its lowest set bit

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/number-of-1-bits)

<a id="q3"></a>
### Q3. Counting Bits

🟢 Warm-up · *DP on i &gt;&gt; 1*

Return an array where ans[i] is the number of 1 bits in i, for every i from 0 to n.

**Examples**

- `n = 2` → `[0, 1, 1]`
- `n = 5` → `[0, 1, 1, 2, 1, 2]`

In [ ]:
def count_bits(n):
    """
    Return an array where ans[i] is the number of 1 bits in i, for every i from
    0 to n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_bits, [
    ((2,), [0, 1, 1]),
    ((5,), [0, 1, 1, 2, 1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count the bits of every i separately.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

i has the same bits as i &gt;&gt; 1 plus its last bit: ans[i] = ans[i &gt;&gt; 1] + (i & 1).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count the bits of every i separately.

```python
def count_bits(n):
    return [bin(i).count("1") for i in range(n + 1)]
```

**Cost:** O(n log n) time, O(1) extra space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** i has the same bits as i &gt;&gt; 1 plus its last bit: ans[i] = ans[i &gt;&gt; 1] + (i & 1).

```python
def count_bits(n):
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i >> 1] + (i & 1)
    return ans
```

**Cost:** O(n) time, O(1) extra space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count the bits of every i separately. That costs O(n log n) time and O(1) extra space."

"I can do better: i has the same bits as i &gt;&gt; 1 plus its last bit: ans[i] = ans[i &gt;&gt; 1] + (i & 1). That's O(n) time and O(1) extra space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 8`

1. ans[1] = ans[0] + 1 = 1
2. ans[2] = ans[1] + 0 = 1
3. ans[3] = ans[1] + 1 = 2
4. ans[4] = ans[2] + 0 = 1
5. ans[5] = ans[2] + 1 = 2
6. ans[6] = ans[3] + 0 = 2
7. ans[7] = ans[3] + 1 = 3
8. ans[8] = ans[4] + 0 = 1

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/counting-bits)

<a id="q4"></a>
### Q4. Reverse Bits

🟢 Warm-up · *Swap halves with masks*

Reverse the bits of a 32-bit unsigned integer.

**Examples**

- `n = 43261596` → `964176192`
- `n = 4294967293` → `3221225471`

In [ ]:
def reverse_bits(n):
    """
    Reverse the bits of a 32-bit unsigned integer.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_bits, [
    ((43261596,), 964176192),
    ((4294967293,), 3221225471),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Read the 32 bits one by one and build the result from the other end.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Swap the two 16-bit halves, then the 8-bit halves inside each, then 4, 2 and 1 — five mask-and-shift steps.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Read the 32 bits one by one and build the result from the other end.

```python
def reverse_bits(n):
    res = 0
    for _ in range(32):
        res = (res << 1) | (n & 1)
        n >>= 1
    return res
```

**Cost:** O(32) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Swap the two 16-bit halves, then the 8-bit halves inside each, then 4, 2 and 1 — five mask-and-shift steps.

```python
def reverse_bits(n):
    n = ((n >> 16) | (n << 16)) & 0xFFFFFFFF
    n = ((n & 0xFF00FF00) >> 8) | ((n & 0x00FF00FF) << 8)
    n = ((n & 0xF0F0F0F0) >> 4) | ((n & 0x0F0F0F0F) << 4)
    n = ((n & 0xCCCCCCCC) >> 2) | ((n & 0x33333333) << 2)
    n = ((n & 0xAAAAAAAA) >> 1) | ((n & 0x55555555) << 1)
    return n
```

**Cost:** O(log 32) = 5 steps time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Read the 32 bits one by one and build the result from the other end. That costs O(32) time and O(1) space."

"I can do better: Swap the two 16-bit halves, then the 8-bit halves inside each, then 4, 2 and 1 — five mask-and-shift steps. That's O(log 32) = 5 steps time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 2952790017`

1. Swap the 16-bit halves
2. Swap bytes inside each half
3. Swap 4-bit groups
4. Swap 2-bit pairs
5. Swap neighbouring bits — fully reversed

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/reverse-bits)

<a id="q5"></a>
### Q5. Power of Two

🟢 Warm-up · *Exactly one set bit*

Return `True` if n is a power of two.

**Examples**

- `n = 1` → `True`
- `n = 16` → `True`
- `n = 3` → `False`

In [ ]:
def is_power_of_two(n):
    """
    Return True if n is a power of two.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_power_of_two, [
    ((1,), True),
    ((16,), True),
    ((3,), False),
    ((0,), False),
    ((-16,), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide by 2 while even; it's a power of two if you reach 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A power of two has exactly one set bit, so n &gt; 0 and n & (n − 1) == 0.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Divide by 2 while even; it's a power of two if you reach 1.

```python
def is_power_of_two(n):
    if n <= 0:
        return False
    while n % 2 == 0:
        n //= 2
    return n == 1
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A power of two has exactly one set bit, so n &gt; 0 and n & (n − 1) == 0.

```python
def is_power_of_two(n):
    return n > 0 and n & (n - 1) == 0
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide by 2 while even; it's a power of two if you reach 1. That costs O(log n) time and O(1) space."

"I can do better: A power of two has exactly one set bit, so n &gt; 0 and n & (n − 1) == 0. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/power-of-two)

<a id="q6"></a>
### Q6. Power of Four

🟢 Warm-up · *One bit, on an even position*

Return `True` if n is a power of four.

**Examples**

- `n = 16` → `True`
- `n = 5` → `False`
- `n = 1` → `True`

In [ ]:
def is_power_of_four(n):
    """
    Return True if n is a power of four.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_power_of_four, [
    ((16,), True),
    ((5,), False),
    ((1,), True),
    ((8,), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Divide by 4 while divisible; check you reach 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

It must be a power of two, and its single bit must be at an even position: n & 0x55555555 ≠ 0 (mask 0101…0101).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Divide by 4 while divisible; check you reach 1.

```python
def is_power_of_four(n):
    if n <= 0:
        return False
    while n % 4 == 0:
        n //= 4
    return n == 1
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** It must be a power of two, and its single bit must be at an even position: n & 0x55555555 ≠ 0 (mask 0101…0101).

```python
def is_power_of_four(n):
    return n > 0 and n & (n - 1) == 0 and n & 0x55555555 != 0
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Divide by 4 while divisible; check you reach 1. That costs O(log n) time and O(1) space."

"I can do better: It must be a power of two, and its single bit must be at an even position: n & 0x55555555 ≠ 0 (mask 0101…0101). That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/power-of-four)

<a id="q7"></a>
### Q7. Hamming Distance

🟢 Warm-up · *Popcount of XOR*

Return the number of bit positions where x and y differ.

**Examples**

- `x = 1, y = 4` → `2`
- `x = 3, y = 1` → `1`

In [ ]:
def hamming_distance(x, y):
    """
    Return the number of bit positions where x and y differ.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hamming_distance, [
    ((1, 4), 2),
    ((3, 1), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Format both as 32-character binary strings and compare characters.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

x ^ y has a 1 exactly where they differ; count its bits with n & (n − 1).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Format both as 32-character binary strings and compare characters.

```python
def hamming_distance(x, y):
    a, b = format(x, "032b"), format(y, "032b")
    return sum(p != q for p, q in zip(a, b))
```

**Cost:** O(32) time, O(32) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** x ^ y has a 1 exactly where they differ; count its bits with n & (n − 1).

```python
def hamming_distance(x, y):
    d, count = x ^ y, 0
    while d:
        d &= d - 1
        count += 1
    return count
```

**Cost:** O(differing bits) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Format both as 32-character binary strings and compare characters. That costs O(32) time and O(32) space."

"I can do better: x ^ y has a 1 exactly where they differ; count its bits with n & (n − 1). That's O(differing bits) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/hamming-distance)

<a id="q8"></a>
### Q8. Sum of All Subset XOR Totals

🟢 Warm-up · *Each bit is in half the subsets*

Return the sum of the XOR of every subset of `nums`.

**Examples**

- `nums = [1, 3]` → `6`
- `nums = [5, 1, 6]` → `28`
- `nums = [3, 4, 5, 6, 7, 8]` → `480`

In [ ]:
def subset_xor_sum(nums):
    """
    Return the sum of the XOR of every subset of nums.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(subset_xor_sum, [
    (([1, 3],), 6),
    (([5, 1, 6],), 28),
    (([3, 4, 5, 6, 7, 8],), 480),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Enumerate all 2ⁿ subsets with a bitmask and XOR each.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A bit that appears in any number is set in the XOR of exactly half of all subsets. So the answer is (OR of all numbers) × 2ⁿ⁻¹.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Enumerate all 2ⁿ subsets with a bitmask and XOR each.

```python
def subset_xor_sum(nums):
    total = 0
    for mask in range(1 << len(nums)):
        x = 0
        for i, v in enumerate(nums):
            if mask >> i & 1:
                x ^= v
        total += x
    return total
```

**Cost:** O(n · 2ⁿ) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A bit that appears in any number is set in the XOR of exactly half of all subsets. So the answer is (OR of all numbers) × 2ⁿ⁻¹.

```python
def subset_xor_sum(nums):
    acc = 0
    for x in nums:
        acc |= x
    return acc << (len(nums) - 1)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Enumerate all 2ⁿ subsets with a bitmask and XOR each. That costs O(n · 2ⁿ) time and O(1) space."

"I can do better: A bit that appears in any number is set in the XOR of exactly half of all subsets. So the answer is (OR of all numbers) × 2ⁿ⁻¹. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/sum-of-all-subset-xor-totals)

<a id="q9"></a>
### Q9. Find the Difference

🟢 Warm-up · *XOR all characters*

`t` is `s` shuffled with one extra letter. Return that letter.

**Examples**

- `s = 'abcd', t = 'abcde'` → `'e'`
- `s = '', t = 'y'` → `'y'`

In [ ]:
def find_the_difference(s, t):
    """
    t is s shuffled with one extra letter. Return that letter.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_the_difference, [
    (('abcd', 'abcde'), 'e'),
    (('', 'y'), 'y'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count letters of both strings and find the one with a higher count in t.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

XOR the codes of every character in s and t; matching letters cancel, leaving the extra one.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count letters of both strings and find the one with a higher count in t.

```python
def find_the_difference(s, t):
    from collections import Counter
    return next(iter(Counter(t) - Counter(s)))
```

**Cost:** O(n) time, O(σ) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** XOR the codes of every character in s and t; matching letters cancel, leaving the extra one.

```python
def find_the_difference(s, t):
    acc = 0
    for ch in s + t:
        acc ^= ord(ch)
    return chr(acc)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count letters of both strings and find the one with a higher count in t. That costs O(n) time and O(σ) space."

"I can do better: XOR the codes of every character in s and t; matching letters cancel, leaving the extra one. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'abcd', t = 'abcde'`

1. XOR in 'a'
2. XOR in 'b'
3. XOR in 'c'
4. XOR in 'd'
5. XOR in 'a'
6. XOR in 'b'
7. XOR in 'c'
8. XOR in 'd'
9. XOR in 'e'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/find-the-difference)

<a id="q10"></a>
### Q10. Minimum Bit Flips to Convert Number

🟢 Warm-up · *Popcount of XOR*

Return the minimum number of bit flips to turn `start` into `goal`.

**Examples**

- `start = 10, goal = 7` → `3`
- `start = 3, goal = 4` → `3`

In [ ]:
def min_bit_flips(start, goal):
    """
    Return the minimum number of bit flips to turn start into goal.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_bit_flips, [
    ((10, 7), 3),
    ((3, 4), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare both numbers bit by bit over 32 positions.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Bits that differ must be flipped: count the 1s in start ^ goal.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Compare both numbers bit by bit over 32 positions.

```python
def min_bit_flips(start, goal):
    return sum(((start >> i) & 1) != ((goal >> i) & 1) for i in range(32))
```

**Cost:** O(32) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Bits that differ must be flipped: count the 1s in start ^ goal.

```python
def min_bit_flips(start, goal):
    d, count = start ^ goal, 0
    while d:
        d &= d - 1
        count += 1
    return count
```

**Cost:** O(differing bits) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare both numbers bit by bit over 32 positions. That costs O(32) time and O(1) space."

"I can do better: Bits that differ must be flipped: count the 1s in start ^ goal. That's O(differing bits) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/minimum-bit-flips-to-convert-number)

<a id="q11"></a>
### Q11. Single Number II

🟡 Interview standard · *Count bits mod 3 (ones / twos)*

Every element appears three times except one, which appears once. Find it in O(1) extra space.

**Examples**

- `nums = [2, 2, 3, 2]` → `3`
- `nums = [0, 1, 0, 1, 0, 1, 99]` → `99`
- `nums = [-2, -2, 1, -2]` → `1`

In [ ]:
def single_number_ii(nums):
    """
    Every element appears three times except one, which appears once. Find it in
    O(1) extra space.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(single_number_ii, [
    (([2, 2, 3, 2],), 3),
    (([0, 1, 0, 1, 0, 1, 99],), 99),
    (([-2, -2, 1, -2],), 1),
    (([-4, -4, -4, -7],), -7),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count occurrences in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

For each bit, the number of times it is set across all numbers, mod 3, is that bit of the answer. 'ones' and 'twos' track bits seen once or twice (mod 3) for all bit positions at once.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count occurrences in a dict.

```python
def single_number_ii(nums):
    count = {}
    for x in nums:
        count[x] = count.get(x, 0) + 1
    return next(x for x, c in count.items() if c == 1)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** For each bit, the number of times it is set across all numbers, mod 3, is that bit of the answer. 'ones' and 'twos' track bits seen once or twice (mod 3) for all bit positions at once.

```python
def single_number_ii(nums):
    ones = twos = 0
    for i, x in enumerate(nums):
        ones = (ones ^ x) & ~twos
        twos = (twos ^ x) & ~ones
    return ones
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count occurrences in a dict. That costs O(n) time and O(n) space."

"I can do better: For each bit, the number of times it is set across all numbers, mod 3, is that bit of the answer. 'ones' and 'twos' track bits seen once or twice (mod 3) for all bit positions at once. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [5, 3, 5, 5]`

1. After 5: bits seen once (mod 3) in ones, twice in twos
2. After 3: bits seen once (mod 3) in ones, twice in twos
3. After 5: bits seen once (mod 3) in ones, twice in twos
4. After 5: bits seen once (mod 3) in ones, twice in twos

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/single-number-ii)

<a id="q12"></a>
### Q12. Single Number III

🟡 Interview standard · *XOR, then split by a differing bit*

Exactly two elements appear once; all others appear twice. Return the two, in any order.

**Examples**

- `nums = [1, 2, 1, 3, 2, 5]` → `[3, 5]`
- `nums = [-1, 0]` → `[-1, 0]`
- `nums = [0, 1]` → `[1, 0]`

In [ ]:
def single_number_iii(nums):
    """
    Exactly two elements appear once; all others appear twice. Return the two,
    in any order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(single_number_iii, [(([1, 2, 1, 3, 2, 5],), [3, 5]), (([-1, 0],), [-1, 0]), (([0, 1],), [1, 0])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count occurrences in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

XOR everything to get a ^ b. Any set bit of it (take the lowest, d = x & −x) is where a and b differ. XOR the numbers with that bit set and those without separately — each group holds one answer.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count occurrences in a dict.

```python
def single_number_iii(nums):
    count = {}
    for x in nums:
        count[x] = count.get(x, 0) + 1
    return [x for x, c in count.items() if c == 1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** XOR everything to get a ^ b. Any set bit of it (take the lowest, d = x & −x) is where a and b differ. XOR the numbers with that bit set and those without separately — each group holds one answer.

```python
def single_number_iii(nums):
    both = 0
    for x in nums:
        both ^= x
    low = both & -both
    a = b = 0
    for x in nums:
        if x & low:
            a ^= x
        else:
            b ^= x
    return [a, b]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count occurrences in a dict. That costs O(n) time and O(n) space."

"I can do better: XOR everything to get a ^ b. Any set bit of it (take the lowest, d = x & −x) is where a and b differ. XOR the numbers with that bit set and those without separately — each group holds one answer. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/single-number-iii)

<a id="q13"></a>
### Q13. Sum of Two Integers

🟡 Interview standard · *XOR = sum without carry, AND = carry*

Return a + b without using + or −.

**Examples**

- `a = 1, b = 2` → `3`
- `a = 2, b = 3` → `5`
- `a = -2, b = 3` → `1`

In [ ]:
def get_sum(a, b):
    """
    Return a + b without using + or −.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(get_sum, [
    ((1, 2), 3),
    ((2, 3), 5),
    ((-2, 3), 1),
    ((-5, -7), -12),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ripple-carry adder: add bit by bit over all 32 positions with a carry bit.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

a ^ b adds without carrying; (a & b) &lt;&lt; 1 is the carry. Repeat until the carry is 0, masking to 32 bits for Python's unbounded ints.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Ripple-carry adder: add bit by bit over all 32 positions with a carry bit.

```python
def get_sum(a, b):
    MASK = 0xFFFFFFFF
    a, b = a & MASK, b & MASK
    res, carry = 0, 0
    for i in range(32):
        x, y = (a >> i) & 1, (b >> i) & 1
        res |= (x ^ y ^ carry) << i
        carry = (x & y) | (carry & (x ^ y))
    return res if res < 2 ** 31 else res - 2 ** 32
```

**Cost:** O(32) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** a ^ b adds without carrying; (a & b) &lt;&lt; 1 is the carry. Repeat until the carry is 0, masking to 32 bits for Python's unbounded ints.

```python
def get_sum(a, b):
    MASK = 0xFFFFFFFF
    a, b = a & MASK, b & MASK
    while b:
        a, b = (a ^ b) & MASK, ((a & b) << 1) & MASK
    return a if a < 2 ** 31 else a - 2 ** 32
```

**Cost:** O(32) worst, usually fewer time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ripple-carry adder: add bit by bit over all 32 positions with a carry bit. That costs O(32) time and O(1) space."

"I can do better: a ^ b adds without carrying; (a & b) &lt;&lt; 1 is the carry. Repeat until the carry is 0, masking to 32 bits for Python's unbounded ints. That's O(32) worst, usually fewer time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `a = 13, b = 11`

1. a = sum without carry, b = the carries shifted left
2. a = sum without carry, b = the carries shifted left
3. a = sum without carry, b = the carries shifted left
4. a = sum without carry, b = the carries shifted left

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/sum-of-two-integers)

<a id="q14"></a>
### Q14. Bitwise AND of Numbers Range

🟡 Interview standard · *Common binary prefix*

Return the AND of every number from `left` to `right` inclusive.

**Examples**

- `left = 5, right = 7` → `4`
- `left = 0, right = 0` → `0`
- `left = 1, right = 1000` → `0`

In [ ]:
def range_bitwise_and(left, right):
    """
    Return the AND of every number from left to right inclusive.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(range_bitwise_and, [
    ((5, 7), 4),
    ((0, 0), 0),
    ((1, 1000), 0),
    ((26, 30), 24),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

AND all numbers in the range one by one.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Any bit that changes somewhere in the range becomes 0. The result is the common prefix of left and right: shift both right until they're equal, then shift back.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** AND all numbers in the range one by one.

```python
def range_bitwise_and(left, right):
    res = left
    for x in range(left + 1, right + 1):
        res &= x
    return res
```

**Cost:** O(right − left) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Any bit that changes somewhere in the range becomes 0. The result is the common prefix of left and right: shift both right until they're equal, then shift back.

```python
def range_bitwise_and(left, right):
    shift = 0
    while left != right:
        left >>= 1
        right >>= 1
        shift += 1
    return left << shift
```

**Cost:** O(log right) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: AND all numbers in the range one by one. That costs O(right − left) time and O(1) space."

"I can do better: Any bit that changes somewhere in the range becomes 0. The result is the common prefix of left and right: shift both right until they're equal, then shift back. That's O(log right) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `left = 26, right = 30`

1. 26 ≠ 30: drop the last bit of both
2. 13 ≠ 15: drop the last bit of both
3. 6 ≠ 7: drop the last bit of both
4. Common prefix 3; shift back by 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/bitwise-and-of-numbers-range)

<a id="q15"></a>
### Q15. Total Hamming Distance

🟡 Interview standard · *Count each bit: ones × zeros*

Return the sum of Hamming distances between all pairs of numbers.

**Examples**

- `nums = [4, 14, 2]` → `6`
- `nums = [4, 14, 4]` → `4`

In [ ]:
def total_hamming_distance(nums):
    """
    Return the sum of Hamming distances between all pairs of numbers.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(total_hamming_distance, [
    (([4, 14, 2],), 6),
    (([4, 14, 4],), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the Hamming distance of every pair.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Look at one bit at a time: if k numbers have it set, it contributes k × (n − k) differing pairs.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Compute the Hamming distance of every pair.

```python
def total_hamming_distance(nums):
    return sum(bin(nums[i] ^ nums[j]).count("1") for i in range(len(nums)) for j in range(i + 1, len(nums)))
```

**Cost:** O(n² · 32) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Look at one bit at a time: if k numbers have it set, it contributes k × (n − k) differing pairs.

```python
def total_hamming_distance(nums):
    n, total = len(nums), 0
    for b in range(32):
        k = sum((x >> b) & 1 for x in nums)
        total += k * (n - k)
        if b < 4:
            pass
    return total
```

**Cost:** O(32 · n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the Hamming distance of every pair. That costs O(n² · 32) time and O(1) space."

"I can do better: Look at one bit at a time: if k numbers have it set, it contributes k × (n − k) differing pairs. That's O(32 · n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [4, 14, 2]`

1. Bit 0: 0 set × 3 unset = 0 pairs
2. Bit 1: 2 set × 1 unset = 2 pairs
3. Bit 2: 2 set × 1 unset = 2 pairs
4. Bit 3: 1 set × 2 unset = 2 pairs

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/total-hamming-distance)

<a id="q16"></a>
### Q16. Divide Two Integers

🟡 Interview standard · *Subtract shifted divisors*

Divide without \*, / or %, truncating toward zero. Clamp the result to 32-bit signed range.

**Examples**

- `dividend = 10, divisor = 3` → `3`
- `dividend = 7, divisor = -3` → `-2`
- `dividend = 1, divisor = 1` → `1`

In [ ]:
def divide(dividend, divisor):
    """
    Divide without *, / or %, truncating toward zero. Clamp the result to 32-bit
    signed range.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(divide, [
    ((10, 3), 3),
    ((7, -3), -2),
    ((1, 1), 1),
    ((-100, 7), -14),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Subtract the divisor repeatedly and count.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Subtract the largest divisor × 2ᵏ that still fits (found by shifting left), add 2ᵏ to the quotient, and repeat on the remainder. Handle the sign separately and clamp −2³¹ / −1.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Subtract the divisor repeatedly and count.

```python
def divide(dividend, divisor):
    neg = (dividend < 0) != (divisor < 0)
    a, b, q = abs(dividend), abs(divisor), 0
    while a >= b:
        a -= b
        q += 1
    q = -q if neg else q
    return max(-2 ** 31, min(2 ** 31 - 1, q))
```

**Cost:** O(quotient) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Subtract the largest divisor × 2ᵏ that still fits (found by shifting left), add 2ᵏ to the quotient, and repeat on the remainder. Handle the sign separately and clamp −2³¹ / −1.

```python
def divide(dividend, divisor):
    if dividend == -2 ** 31 and divisor == -1:
        return 2 ** 31 - 1
    neg = (dividend < 0) != (divisor < 0)
    a, b, q = abs(dividend), abs(divisor), 0
    while a >= b:
        k = 0
        while a >= b << (k + 1):
            k += 1
        a -= b << k
        q += 1 << k
    return -q if neg else q
```

**Cost:** O(log² n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Subtract the divisor repeatedly and count. That costs O(quotient) time and O(1) space."

"I can do better: Subtract the largest divisor × 2ᵏ that still fits (found by shifting left), add 2ᵏ to the quotient, and repeat on the remainder. Handle the sign separately and clamp −2³¹ / −1. That's O(log² n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `dividend = 100, divisor = 7`

1. Largest fit: 7 × 2^3 = 56; remainder 44
2. Largest fit: 7 × 2^2 = 28; remainder 16
3. Largest fit: 7 × 2^1 = 14; remainder 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/divide-two-integers)

<a id="q17"></a>
### Q17. Minimum Flips to Make a OR b Equal to c

🟡 Interview standard · *Per-bit cases*

Return the fewest single-bit flips in a and b so that a | b == c.

**Examples**

- `a = 2, b = 6, c = 5` → `3`
- `a = 4, b = 2, c = 7` → `1`
- `a = 1, b = 2, c = 3` → `0`

In [ ]:
def min_flips(a, b, c):
    """
    Return the fewest single-bit flips in a and b so that a | b == c.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_flips, [
    ((2, 6, 5), 3),
    ((4, 2, 7), 1),
    ((1, 2, 3), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare padded binary strings of a, b and c position by position.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

If c's bit is 1 and both are 0 → 1 flip. If c's bit is 0 → flip every 1 in a and b. In masks: popcount((a | b) ^ c) + popcount(a & b & ~c).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Compare padded binary strings of a, b and c position by position.

```python
def min_flips(a, b, c):
    x, y, z = format(a, "032b"), format(b, "032b"), format(c, "032b")
    flips = 0
    for p, q, r in zip(x, y, z):
        if r == "1":
            flips += p == "0" and q == "0"
        else:
            flips += (p == "1") + (q == "1")
    return flips
```

**Cost:** O(32) time, O(32) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** If c's bit is 1 and both are 0 → 1 flip. If c's bit is 0 → flip every 1 in a and b. In masks: popcount((a | b) ^ c) + popcount(a & b & ~c).

```python
def min_flips(a, b, c):
    wrong = (a | b) ^ c
    double = a & b & ~c
    return bin(wrong).count("1") + bin(double).count("1")
```

**Cost:** O(1) with popcount time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare padded binary strings of a, b and c position by position. That costs O(32) time and O(32) space."

"I can do better: If c's bit is 1 and both are 0 → 1 flip. If c's bit is 0 → flip every 1 in a and b. In masks: popcount((a | b) ^ c) + popcount(a & b & ~c). That's O(1) with popcount time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/minimum-flips-to-make-a-or-b-equal-to-c)

<a id="q18"></a>
### Q18. Count Triplets That Can Form Two Arrays of Equal XOR

🟡 Interview standard · *Prefix XOR + index sums*

Count triplets i &lt; j ≤ k where XOR(arr[i..j−1]) == XOR(arr[j..k]).

**Examples**

- `arr = [2, 3, 1, 6, 7]` → `4`
- `arr = [1, 1, 1, 1, 1]` → `10`

In [ ]:
def count_triplets(arr):
    """
    Count triplets i < j ≤ k where XOR(arr[i..j−1]) == XOR(arr[j..k]).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_triplets, [
    (([2, 3, 1, 6, 7],), 4),
    (([1, 1, 1, 1, 1],), 10),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every (i, j, k) and compute both XORs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Both halves are equal exactly when XOR(arr[i..k]) = 0, i.e. pre[i] == pre[k + 1]; then every j in (i, k] works, giving k − i triplets. Keep, per prefix value, how many times it appeared and the sum of their indices.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every (i, j, k) and compute both XORs.

```python
def count_triplets(arr):
    n, count = len(arr), 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j, n):
                a = b = 0
                for t in range(i, j):
                    a ^= arr[t]
                for t in range(j, k + 1):
                    b ^= arr[t]
                count += a == b
    return count
```

**Cost:** O(n³) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Both halves are equal exactly when XOR(arr[i..k]) = 0, i.e. pre[i] == pre[k + 1]; then every j in (i, k] works, giving k − i triplets. Keep, per prefix value, how many times it appeared and the sum of their indices.

```python
def count_triplets(arr):
    cnt, idx_sum = {0: 1}, {0: 0}
    pre = ans = 0
    for k, x in enumerate(arr):
        pre ^= x
        ans += cnt.get(pre, 0) * k - idx_sum.get(pre, 0)
        cnt[pre] = cnt.get(pre, 0) + 1
        idx_sum[pre] = idx_sum.get(pre, 0) + k + 1
    return ans
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every (i, j, k) and compute both XORs. That costs O(n³) time and O(1) space."

"I can do better: Both halves are equal exactly when XOR(arr[i..k]) = 0, i.e. pre[i] == pre[k + 1]; then every j in (i, k] works, giving k − i triplets. Keep, per prefix value, how many times it appeared and the sum of their indices. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `arr = [2, 3, 1, 6, 7]`

1. prefix XOR through index 0 = 2; earlier equal prefixes add their k − i
2. prefix XOR through index 1 = 1; earlier equal prefixes add their k − i
3. prefix XOR through index 2 = 0; earlier equal prefixes add their k − i
4. prefix XOR through index 3 = 6; earlier equal prefixes add their k − i
5. prefix XOR through index 4 = 1; earlier equal prefixes add their k − i

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/count-triplets-that-can-form-two-arrays-of-equal-xor)

<a id="q19"></a>
### Q19. XOR Queries of a Subarray

🟡 Interview standard · *Prefix XOR*

For each query [l, r], return arr[l] ^ … ^ arr[r].

**Examples**

- `arr = [1, 3, 4, 8], queries = [[0, 1], [1, 2], [0, 3], [3, 3]]` → `[2, 7, 14, 8]`
- `arr = [4, 8, 2, 10], queries = [[2, 3], [1, 3], [0, 0], [0, 3]]` → `[8, 0, 4, 4]`

In [ ]:
def xor_queries(arr, queries):
    """
    For each query [l, r], return arr[l] ^ … ^ arr[r].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(xor_queries, [
    (([1, 3, 4, 8], [[0, 1], [1, 2], [0, 3], [3, 3]]), [2, 7, 14, 8]),
    (([4, 8, 2, 10], [[2, 3], [1, 3], [0, 0], [0, 3]]), [8, 0, 4, 4]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

XOR the range from scratch for each query.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

pre[i] = XOR of the first i elements. Since x ^ x = 0, xor(l..r) = pre[r + 1] ^ pre[l].

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** XOR the range from scratch for each query.

```python
def xor_queries(arr, queries):
    res = []
    for l, r in queries:
        x = 0
        for i in range(l, r + 1):
            x ^= arr[i]
        res.append(x)
    return res
```

**Cost:** O(n · q) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** pre[i] = XOR of the first i elements. Since x ^ x = 0, xor(l..r) = pre[r + 1] ^ pre[l].

```python
def xor_queries(arr, queries):
    pre = [0]
    for x in arr:
        pre.append(pre[-1] ^ x)
    res = []
    for l, r in queries:
        res.append(pre[r + 1] ^ pre[l])
    return res
```

**Cost:** O(n + q) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: XOR the range from scratch for each query. That costs O(n · q) time and O(1) space."

"I can do better: pre[i] = XOR of the first i elements. Since x ^ x = 0, xor(l..r) = pre[r + 1] ^ pre[l]. That's O(n + q) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `arr = [1, 3, 4, 8], queries = [[0, 1], [1, 2], [0, 3], [3, 3]]`

1. Prefix XOR: pre[i] = arr[0] ^ … ^ arr[i − 1]
2. [0, 1] → pre[2] ^ pre[0] = 2 ^ 0 = 2
3. [1, 2] → pre[3] ^ pre[1] = 6 ^ 1 = 7
4. [0, 3] → pre[4] ^ pre[0] = 14 ^ 0 = 14
5. [3, 3] → pre[4] ^ pre[3] = 14 ^ 6 = 8

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/xor-queries-of-a-subarray)

<a id="q20"></a>
### Q20. Gray Code

🟡 Interview standard · *i ^ (i &gt;&gt; 1)*

Return any sequence of all 2ⁿ n-bit numbers starting at 0 where consecutive numbers (including last → first) differ by exactly one bit.

**Examples**

- `n = 2` → `[0, 1, 3, 2]`
- `n = 1` → `[0, 1]`
- `n = 3` → `[0, 1, 3, 2, 6, 7, 5, 4]`

In [ ]:
def gray_code(n):
    """
    Return any sequence of all 2ⁿ n-bit numbers starting at 0 where consecutive
    numbers (including last → first) differ by exactly one bit.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(gray_code, [((2,), [0, 1, 3, 2]), ((1,), [0, 1]), ((3,), [0, 1, 3, 2, 6, 7, 5, 4])], verify=lambda a, out: len(out) == 1 << a[0] and out[0] == 0 and len(set(out)) == len(out) and all(bin(out[i] ^ out[(i + 1) % len(out)]).count("1") == 1 for i in range(len(out))))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Backtracking: from the current number try flipping each bit to an unused number until all 2ⁿ are used.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The i-th Gray code is i ^ (i &gt;&gt; 1): moving from i to i + 1 changes this value in exactly one bit.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Backtracking: from the current number try flipping each bit to an unused number until all 2ⁿ are used.

```python
def gray_code(n):
    seq, used = [0], {0}

    def go():
        if len(seq) == 1 << n:
            return True
        for b in range(n):
            nxt = seq[-1] ^ (1 << b)
            if nxt not in used:
                used.add(nxt)
                seq.append(nxt)
                if go():
                    return True
                seq.pop()
                used.remove(nxt)
        return False
    go()
    return seq
```

**Cost:** O(n · 2ⁿ) with backtracking, can be worse time, O(2ⁿ) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The i-th Gray code is i ^ (i &gt;&gt; 1): moving from i to i + 1 changes this value in exactly one bit.

```python
def gray_code(n):
    res = []
    for i in range(1 << n):
        res.append(i ^ (i >> 1))
    return res
```

**Cost:** O(2ⁿ) time, O(1) extra space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Backtracking: from the current number try flipping each bit to an unused number until all 2ⁿ are used. That costs O(n · 2ⁿ) with backtracking, can be worse time and O(2ⁿ) space."

"I can do better: The i-th Gray code is i ^ (i &gt;&gt; 1): moving from i to i + 1 changes this value in exactly one bit. That's O(2ⁿ) time and O(1) extra space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 2`

1. i = 0 → 0 ^ 0 = 0
2. i = 1 → 1 ^ 0 = 1
3. i = 2 → 2 ^ 1 = 3
4. i = 3 → 3 ^ 1 = 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/bits/gray-code)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. How do you check whether a number is a power of two?</b></summary>

`n > 0 and n & (n − 1) == 0`. A power of two has exactly one set bit; subtracting 1 turns it off and sets all lower bits, so the AND is 0.

</details>

<details>
<summary><b>2. Why does n & (n − 1) remove the lowest set bit?</b></summary>

Subtracting 1 flips the lowest 1-bit to 0 and every 0-bit below it to 1, leaving higher bits unchanged. ANDing with n keeps the higher bits and clears everything from the lowest set bit down.

</details>

<details>
<summary><b>3. What is two's complement and why is it used?</b></summary>

Negative x is stored as (bitwise NOT of x) + 1. It has a single zero and lets the same adder circuit handle signed and unsigned arithmetic, so CPUs use it universally.

</details>

<details>
<summary><b>4. How do you find the element that appears once when every other appears twice?</b></summary>

XOR all elements. Equal pairs cancel to 0 (x ^ x = 0), and 0 ^ y = y, leaving the unique element. O(n) time, O(1) space.

</details>

<details>
<summary><b>5. Left shift vs right shift?</b></summary>

`x << k` moves bits left, filling with zeros — multiplies by 2ᵏ. `x >> k` moves bits right — floor-divides by 2ᵏ. Arithmetic right shift copies the sign bit; logical right shift (Java's `>>>`) fills with zeros.

</details>

<details>
<summary><b>6. How do you count set bits efficiently?</b></summary>

Brian Kernighan's loop (`n &= n − 1` until 0) runs once per set bit. Built-ins: `bin(n).count('1')` or `n.bit_count()`. For 0..n, use DP: `bits[i] = bits[i >> 1] + (i & 1)`.

</details>

<details>
<summary><b>7. Why do Python bit problems sometimes need & 0xFFFFFFFF?</b></summary>

Python ints are unbounded, so negatives behave as if they had infinitely many leading 1s and loops like carry propagation never terminate. Masking to 32 bits simulates fixed-width integers; convert back with `x − 2**32` when the top bit is set.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Binary numbers
- [ ] I can explain: The six operators
- [ ] I can explain: Working with one bit
- [ ] I can explain: XOR, the cancelling operator
- [ ] I can explain: Counting and comparing bits
- [ ] I can explain: Negative numbers and arithmetic with bits
- [ ] I can explain: Per-bit thinking, prefix XOR and subsets

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Divide Two Integers
- [ ] [Q17](#q17) Minimum Flips to Make a OR b Equal to c
- [ ] [Q18](#q18) Count Triplets That Can Form Two Arrays of Equal XOR
- [ ] [Q19](#q19) XOR Queries of a Subarray
- [ ] [Q20](#q20) Gray Code

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.